# Data Description

- Background: 這是一份用來練習的假訂單資料，共有 122 列、5 個欄位。裡面刻意放了重複紀錄、缺值和負單價，方便練習清理。後續會以商品類別與日期，合併 [類別表](demo_categories_documentation.ipynb) 和 [日期表](demo_calendar_documentation.ipynb)。
- date received: 20261013（資料生成日期）
- Path to data file: 由 [demo_generate_orders.py](../code/demo_generate_orders.py) 產生，沒有外部下載來源。在專案根目錄執行 `python data/code/demo_generate_orders.py` 即可建立相同資料。
- Unit of observation: 每列是一筆訂單紀錄，同一筆訂單可能重複出現。
- Sample period: 設定為 2026-09-01 至 2026-09-30。
- Known issues:
    - 有 2 列完整重複的紀錄。
    - `quantity` 有 3 個缺值，`unit_price_twd` 有 2 個負值。
    - 部分商品類別前後有空白，例如 `" A "` 和 `"A"` 會被當成不同類別。
- Definition for each variable:
    - `order_id`：訂單編號，用來辨識重複訂單。
    - `order_date`：訂單日期，格式為 YYYY-MM-DD。
    - `category`：商品類別，以 A、B、C 表示。
    - `quantity`：購買件數。
    - `unit_price_twd`：每件商品的單價，單位為新臺幣元。
- Data path: [data/raw/demo_orders.csv](demo_orders.csv)
- Next processing notebook: [demo_orders_processing.ipynb](../code/demo_orders_processing.ipynb)

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 找到專案資料夾，從不同位置開啟 notebook 都能讀到資料。
project_root = Path.cwd().resolve()
while not ((project_root / "AGENTS.md").is_file() and (project_root / "templates").is_dir()):
    if project_root == project_root.parent:
        raise FileNotFoundError("請從專案資料夾內開啟 notebook。")
    project_root = project_root.parent

## Change for Your Own Data

這裡設定要讀取的 CSV。之後換資料時，修改下方檔案路徑，再確認欄位名稱與內容。

In [2]:
data_path = project_root / "data/raw/demo_orders.csv"

## Summary

先確認資料的大小與欄位型態，再查看前幾筆紀錄。這裡只檢查資料，不進行刪除或補值。

In [3]:
df = pd.read_csv(data_path)
print(f"共 {len(df)} 列、{len(df.columns)} 欄")
print(f"完整重複：{df.duplicated().sum()} 列")
print(f"購買件數缺值：{df['quantity'].isna().sum()} 筆")
print(f"負單價：{(df['unit_price_twd'] < 0).sum()} 筆")


共 122 列、5 欄
完整重複：2 列
購買件數缺值：3 筆
負單價：2 筆


### Sample

以下列出前 10 筆。第 9 筆訂單的購買件數為缺值，後續計算訂單金額前需要先處理。

In [4]:
df.head(10)

,order_id,order_date,category,quantity,unit_price_twd
0,D0001,2026-09-01,A,1.0,80
1,D0002,2026-09-02,B,3.0,97
2,D0003,2026-09-03,C,5.0,114
3,D0004,2026-09-04,A,2.0,131
4,D0005,2026-09-05,B,4.0,148
5,D0006,2026-09-06,C,1.0,165
6,D0007,2026-09-07,A,3.0,182
7,D0008,2026-09-08,B,5.0,199
8,D0009,2026-09-09,C,NaN,216
9,D0010,2026-09-10,A,4.0,233


### Summary Stats

`count` 是各欄位的非缺值筆數，因此購買件數只有 119 筆，其餘欄位各有 122 筆。商品類別目前出現 6 種寫法，是因為部分值帶有空白；清理後應只剩 A、B、C 三類。

In [5]:
print(f"原始資料摘要（N={len(df)}；count 為非缺值筆數）")
display(df.describe(include="all"))

column_info = pd.DataFrame({
    "型態": df.dtypes.astype(str),
    "缺值數": df.isna().sum(),
})
print("欄位型態與缺值數")
display(column_info)


原始資料摘要（N=122；count 為非缺值筆數）


,order_id,order_date,category,quantity,unit_price_twd
count,122,122,122,119.000000,122.000000
unique,120,30,6,NaN,NaN
top,D0001,2026-09-01,B,NaN,NaN
freq,2,5,38,NaN,NaN
mean,NaN,NaN,NaN,3.008403,174.450820
std,NaN,NaN,NaN,1.429091,73.750428
min,NaN,NaN,NaN,1.000000,-100.000000
25%,NaN,NaN,NaN,2.000000,114.000000
50%,NaN,NaN,NaN,3.000000,182.000000
75%,NaN,NaN,NaN,4.000000,233.000000


欄位型態與缺值數


,型態,缺值數
order_id,object,0
order_date,object,0
category,object,0
quantity,float64,3
unit_price_twd,int64,0
